In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

print("Starting Silver → Gold Transformation...")

# Configuration
SILVER_TABLE = "silver.customerchurn"
GOLD_TABLE = "gold.customerChurn"
BATCH_SIZE = 1000

  # Step 1: Read silver data (incremental)
df_silver = spark.sql(f"""SELECT * FROM {SILVER_TABLE}""")
display(df_silver)

In [ ]:
row_count_before = df_silver.count()
print(f"Processing {row_count_before} records from silver layer...")
    
if row_count_before == 0:
    print("⚠ No new data to process. Exiting.")
    dbutils.notebook.exit("NO_DATA")

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

print("Starting Silver → Gold Transformation...")

# Configuration
SILVER_TABLE = "silver.customerchurn"
BATCH_SIZE = 10000

  # Step 1: Read silver data (incremental)
df_silver = spark.sql(f"""SELECT * FROM {SILVER_TABLE}""")
display(df_silver)

row_count_before = df_silver.count()
print(f"Processing {row_count_before} records from silver layer...")
    
if row_count_before == 0:
    print("⚠ No new data to process. Exiting.")
    dbutils.notebook.exit("NO_DATA")

#  # Step 2: Aggregate data

df_gold = (df_silver
    .orderBy(
        col("cr8b0_internetservicetype")
    )
    .groupBy(
        col("cr8b0_internetservicetype"),
        col("cr8b0_paymentmethod")
    )
    .agg(
        count("*").alias("transaction_count"),
        sum(col("cr8b0_totalcharges")).alias("TotalSumCharges")
    )
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

row_count_gold = df_gold.count()
print(f"Aggregated to {row_count_gold} gold records")


# Step 3: Write to gold (append, partitioned by load_date)
df_gold.write.format("delta").mode("overwrite").saveAsTable("gold.CustomerChurn")

display(df_gold)